The script acts as a Python script job designed to simulate external data fetching via Kagglehub APIs. Its primary role is to land raw data into External volumes for landing data, ensuring that high-overhead formats like XML are generated efficiently using Spark's distributed processing.

In [0]:
import os
import shutil
import kagglehub
import pandas as pd
import gc
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window



# --- 1. CONFIGURATION ---
# The Databricks Secret Scope containing Kaggle credentials
try:
    from schecrets import KAGGLE_SCOPE, DATASET_HANDLE, VOLUME_BASE_PATH, SIZE_LIMIT_BYTES
except ImportError:
    KAGGLE_SCOPE = "KaggleCreds" 
    # The unique identifier for the Zillow dataset on Kaggle
    DATASET_HANDLE = "zillow/zecon"
    # Path to the Unity Catalog Volume for landing raw files
    VOLUME_BASE_PATH = "/Volumes/data_landing/data_raw"
    # Threshold for chunking logic (500MB)
    SIZE_LIMIT_BYTES = 500 * 1024 * 1024 

def initialize_kaggle_auth(scope):
    """
    Retrieves Kaggle API credentials from Databricks Secrets 
    and injects them into the OS environment for kagglehub/kaggle-api.
    """
    os.environ['KAGGLE_USERNAME'] = dbutils.secrets.get(scope=scope, key="username")
    os.environ['KAGGLE_KEY'] = dbutils.secrets.get(scope=scope, key="key")

def generate_xml_spark_chunk(volume_source_path, dest_dir, start_offset, dataset_name):
    """
    Transforms a CSV source into an XML chunk using Spark.
    Designed to handle high-memory overhead by utilizing Spark's distributed processing.
    """
    print(f"   - Starting Spark XML generation (Chunk 4)...")
    
    # 1. READ: Load CSV with specific encoding to handle special characters
    # ISO-8859-1 is used to prevent 'utf-8' codec errors in housing descriptions
    df = spark.read.option("header", "true") \
                  .option("encoding", "ISO-8859-1") \
                  .csv(volume_source_path)
    
    # 2. SCHEMA FIX: Cast all columns to String to ensure XML compatibility 
    # and prevent inference errors during serialization
    for col_name in df.columns:
        df = df.withColumn(col_name, F.col(col_name).cast(StringType()))

    # 3. WINDOWING: Assign row numbers to implement the offset/chunking logic
    # monotonically_increasing_id provides a fast, stable row ID for large datasets
    window_spec = Window.orderBy(F.monotonically_increasing_id())
    chunk4_df = df.withColumn("row_num", F.row_number().over(window_spec)) \
        .filter(F.col("row_num") > start_offset) \
        .drop("row_num")
    
    # 4. AUDIT COLS: Add metadata for traceability (Load date and source file name)
    chunk4_df = chunk4_df \
        .withColumn("load_dt", F.current_timestamp().cast(StringType())) \
        .withColumn("source", F.lit(f"{dataset_name}.csv"))

    # 5. WRITE: Serialize to XML format
    # coalesce(1) ensures we generate a single file for the specific chunk
    temp_xml_path = f"{dest_dir}/temp_xml_{dataset_name}"
    (chunk4_df.coalesce(1).write
     .format("xml")
     .option("rootTag", "ZillowData") # Root XML element
     .option("rowTag", "Record")      # Repeating element for each row
     .mode("overwrite")
     .save(temp_xml_path))

    # 6. CLEANUP: Move the Spark part-file to a human-readable name and delete temp folder
    try:
        files = [f for f in os.listdir(temp_xml_path) if f.startswith("part-") and f.endswith(".xml")]
        if files:
            shutil.move(os.path.join(temp_xml_path, files[0]), f"{dest_dir}/chunk4.xml")
            shutil.rmtree(temp_xml_path)
            print(f"   - Successfully dumped chunk4.xml")
    except Exception as e:
        print(f"   - XML move failed: {str(e)}")

The provided functions manage the transformation of raw Kaggle data into a structured landing zone. By diversifying output formats (CSV, JSON, XML), the system ensures flexibility for different downstream consumers in the **Medallion Architecture**.|

In [0]:
def process_large_file_split(volume_source_path, dest_dir, total_rows, dataset_name):
    """
    Splits massive CSV files into 4 chunks (CSV, CSV, JSON, and XML).
    Includes an encoding fix to handle special characters found in Zillow data.
    """
    # Ensure the destination directory exists on the local DBFS/Volume
    os.makedirs(dest_dir, exist_ok=True)
    
    # Calculate row-based splits based on percentages
    c1_size = int(total_rows * 0.40) # First 40%
    c2_size = int(total_rows * 0.40) # Next 40%
    c3_size = int(total_rows * 0.10) # Next 10% (The remaining 10% is handled by Spark)

    # Loop through the first three chunks
    for i, (skip, nrows, fmt, name) in enumerate([
        (0, c1_size, 'csv', 'chunk1.csv'),
        (c1_size, c2_size, 'csv', 'chunk2.csv'),
        (c1_size + c2_size, c3_size, 'json', 'chunk3.json')
    ]):
        # Calculate which rows to skip (handling header offset)
        skip_range = range(1, skip + 1) if skip > 0 else None
        target = f"{dest_dir}/{name}"
        
        # FIX: ISO-8859-1 encoding is critical for housing data that may contain 
        # symbols or accented characters not supported by standard UTF-8.
        # dtype=str prevents Pandas from guessing types, which saves RAM.
        df = pd.read_csv(volume_source_path, skiprows=skip_range, nrows=nrows, 
                         low_memory=False, dtype=str, encoding='ISO-8859-1')
        
        # Add metadata for the Gold layer auditing
        df['load_dt'] = pd.Timestamp.now()
        df['source'] = f"{dataset_name}.csv"
        
        # Export based on the requested format
        if fmt == 'csv': 
            df.to_csv(target, index=False)
        else: 
            df.to_json(target, orient='records')
            
        print(f"  - Generated {name}")
        
        # Force memory cleanup: delete dataframe and trigger Python garbage collector
        del df
        gc.collect()
    
    # HANDOFF: The final 10% is processed via Spark to generate XML
    # This prevents the Pandas driver from crashing on the final rows of huge files.
    generate_xml_spark_chunk(volume_source_path, dest_dir, (c1_size + c2_size + c3_size), dataset_name)

def process_small_file_direct(volume_source_path, dest_volume_dir, dataset_name):
    """
    Processes smaller files that do not require splitting.
    Includes the same encoding and auditing logic as the large file process.
    """
    print(f"  - Finalizing small file: {dataset_name}")
    
    # Standard read with encoding fix
    df = pd.read_csv(volume_source_path, low_memory=False, dtype=str, encoding='ISO-8859-1')
    
    # Add audit columns
    df['load_dt'] = pd.Timestamp.now()
    df['source'] = f"{dataset_name}.csv"
    
    # Define final landing path in the Volume
    final_path = f"{dest_volume_dir}/{dataset_name}.csv"
    df.to_csv(final_path, index=False)
    
    # Cleanup: remove the temporary raw file if it has been moved/processed
    if volume_source_path != final_path:
        os.remove(volume_source_path)
    
    # Memory cleanup
    del df
    gc.collect()

The **run_data_chunking_job** function serves as the primary orchestration layer for the initial data ingestion phase of the pipeline. It manages the end-to-end lifecycle of raw data, from secure external fetching to landing in distributed volumes for further processing.

In [0]:
def run_data_chunking_job():
    """
    Main orchestration function that manages the download, migration, 
    and decision logic for the Zillow data pipeline.
    """
    # 1. AUTH: Initialize secure connection to Kaggle using Databricks Secrets
    initialize_kaggle_auth(KAGGLE_SCOPE)
    
    # 2. DOWNLOAD: Fetch the dataset to the local driver node cache
    print(f"Downloading dataset: {DATASET_HANDLE}")
    local_cache_path = kagglehub.dataset_download(DATASET_HANDLE)
    
    # Filter for CSV files only, ignoring metadata or documentation files
    all_files = [f for f in os.listdir(local_cache_path) if f.lower().endswith('.csv')]
    
    for fname in all_files:
        # Prepare paths for migration to Unity Catalog Volumes
        local_path = os.path.join(local_cache_path, fname)
        clean_name = fname.lower().replace('.csv', '')
        dest_volume_dir = f"{VOLUME_BASE_PATH}/{clean_name}"
        
        # Create directory structure within the Volume
        os.makedirs(dest_volume_dir, exist_ok=True)
        
        # 3. MIGRATE: Move file from local cache to a temp location in the Volume
        intermediate_vol_path = f"{dest_volume_dir}/raw_temp_{fname.lower()}"
        shutil.move(local_path, intermediate_vol_path)
        
        # 4. DECIDE: Check file size to determine the processing path
        fsize = os.path.getsize(intermediate_vol_path)

        if fsize > SIZE_LIMIT_BYTES:
            print(f"\n[LARGE FILE] Splitting {fname} (Size: {fsize/(1024**2):.2f} MB)...")
            
            # FIX: Memory-efficient line counting. 
            # encoding='ISO-8859-1' and errors='ignore' handle non-standard real estate text.
            with open(intermediate_vol_path, 'r', encoding='ISO-8859-1', errors='ignore') as f:
                # Subtract 1 to account for the header row
                total_rows = sum(1 for _ in f) - 1
            
            # Trigger the multi-format chunking process (CSV, JSON, XML)
            process_large_file_split(intermediate_vol_path, f"{dest_volume_dir}/chunks", total_rows, clean_name)
            
            # Cleanup the temporary large file to save space in the Volume
            os.remove(intermediate_vol_path) 
        else:
            # 5. DIRECT: Process smaller files without splitting
            print(f"\n[SMALL FILE] Processing {fname}...")
            process_small_file_direct(intermediate_vol_path, dest_volume_dir, clean_name)

    print("\n[SUCCESS] Pipeline Finished.")

if __name__ == "__main__":
    run_data_chunking_job()